# 買い目フィルタ（人気×RL乖離マトリクス）の大規模out-of-sample検証

## 目的
現在の買い目フィルタは **4開催日・76点** しか検証できておらず、うち2日がマイナス
（7/11は16点買って0的中）。通算ROI 182.8%はプラスだが確証には程遠い。

本ノートは history.db の過去データを使い、**約2,000レース規模** で検証する。

| | 規模 |
|---|---|
| 学習期（〜2025-12） | 約3,400レース |
| **検証期（2026-01〜）** | **約1,900レース / 約2万頭** |

## 🔴 最重要：なぜ「再学習」が必須か
現行の本番モデルは **2026年6月までのデータで学習済み**。そのまま過去レースに
当てると「モデルが答えを覚えている状態」でのテストになり、結果は必ず良く出るが
実運用の性能とは無関係になる（look-ahead bias）。

そこで **2025年末までのデータだけで学習し直したモデル** を作り、
2026年を純粋なout-of-sampleとして評価する。

## ⚠ 本番ファイルは一切上書きしない
検証用モデルは `data/backtest/` に保存する。`xgb_fukusho_model.pkl` 等の
本番ファイルには触れないので、週末の運用に影響しない。

## 実行方法
**セル1から順に最後まで実行するだけ。** 所要時間の目安は合計40〜80分。

| セル | 内容 | 目安時間 |
|---|---|---|
| 1 | セットアップ（Drive接続・最新コード取得） | 1分 |
| 2 | 学習データ生成（history.dbから特徴量計算） | 20〜50分 |
| 3 | 検証用モデルの学習（2025年末まで） | 5〜15分 |
| 4 | 2026年の全レースを推論 → rl_rank算出 | 1〜3分 |
| 5 | **マトリクス作成と検証（結果はここ）** | 1分 |
| 6 | より良いフィルタの探索 | 1分 |

## セル1: セットアップ

Driveを接続し、GitHubから最新のコードを取得する。

In [ ]:
import os, sys, subprocess, urllib.request, time

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
assert os.path.exists(BASE_DIR), f'BASE_DIRが見つかりません: {BASE_DIR}'
os.chdir(BASE_DIR)
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# 検証用の出力先（本番ファイルと分離）
OUT_DIR = os.path.join(BASE_DIR, 'data', 'backtest')
os.makedirs(OUT_DIR, exist_ok=True)

# ── GitHubから最新コードを取得 ────────────────────────────────
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
FILES = [
    'src/tools/__init__.py', 'src/tools/build_training_data.py', 'src/tools/train_xgb.py',
    'src/features/engine.py', 'src/features/speed_index.py', 'src/features/horse_type.py',
    'src/features/error_tags.py', 'src/features/shap_explain.py',
    'src/utils/config.py', 'src/utils/db.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py', 'src/betting/ev_filter.py',
    'src/betting/app_json.py', 'src/betting/rank_matrix_filter.py',
    'data/course_profiles.json', 'data/course_distance_profiles.json',
    'data/note_schema.json',
]
for rel in FILES:
    dest = os.path.join(BASE_DIR, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    except Exception as e:
        print(f'  ⚠ 取得失敗（既存ファイルを使用）: {rel} — {e}')
for key in [k for k in list(sys.modules) if k.startswith('src')]:
    del sys.modules[key]

# ── データ確認 ───────────────────────────────────────────────
import sqlite3
HIST = os.path.join(BASE_DIR, 'data', 'history.db')
assert os.path.exists(HIST), f'history.dbが見つかりません: {HIST}'
_c = sqlite3.connect(HIST)
_n, _mn, _mx = _c.execute('SELECT COUNT(*), MIN(date), MAX(date) FROM race_history').fetchone()
_h = _c.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
_c.close()
print(f'\n✅ セットアップ完了')
print(f'   history.db : {_n:,}レース / {_h:,}出走')
print(f'   期間       : {_mn} 〜 {_mx}')
print(f'   出力先     : {OUT_DIR}')

# 学習期/検証期の境界（ここを変えれば期間を調整できる）
TRAIN_END  = '2025-12-31'
TEST_START = '2026-01-01'
print(f'\n   学習期: 〜{TRAIN_END} / 検証期: {TEST_START}〜')

## セル2: 学習データ生成

`history.db` から全レース・全馬の特徴量を計算して `horse_features.csv` を作る。

**⚠ ここが一番時間がかかる（20〜50分）。** 途中で切れた場合はこのセルだけ再実行すればよい。

既に最新の `horse_features.csv` がある場合はスキップできる（下の `FORCE_REBUILD` を `False` に）。

In [ ]:
FORCE_REBUILD = True   # 既存CSVを使い回すなら False

import pandas as pd
CSV_PATH = os.path.join(BASE_DIR, 'data', 'horse_features.csv')

_need = FORCE_REBUILD or not os.path.exists(CSV_PATH)
if not _need:
    _tmp = pd.read_csv(CSV_PATH, nrows=1)
    if 'f_popularity' not in _tmp.columns:
        print('⚠ 既存CSVに f_popularity が無い（古い形式）→ 再生成します')
        _need = True

if _need:
    t0 = time.time()
    from src.features.engine import init_engine
    from src.tools.build_training_data import build_training_data
    init_engine(BASE_DIR)
    build_training_data(BASE_DIR)
    print(f'\n⏱ 所要 {(time.time()-t0)/60:.1f} 分')
else:
    print('既存の horse_features.csv を使用します')

df = pd.read_csv(CSV_PATH)
print(f'\n✅ 学習データ: {len(df):,}行 × {len(df.columns)}列')
print(f'   期間: {df["date"].min()} 〜 {df["date"].max()}')
assert 'f_popularity' in df.columns, 'f_popularity が無い。build_training_data の実行を確認'

n_tr = (df['date'] <= TRAIN_END).sum()
n_te = (df['date'] >= TEST_START).sum()
print(f'   学習期: {n_tr:,}行 / 検証期: {n_te:,}行')
assert n_te > 5000, f'検証期のデータが少なすぎます({n_te}行)。TEST_STARTを見直してください'

## セル3: 検証用モデルの学習（2025年末まで）

本番と**同じ設定**（残差学習・同じハイパーパラメータ）で、
**2025年末までのデータだけ**を使って学習する。

本番の `train_xgb()` は本番ファイルを上書きする可能性があるため、
ここでは同等の処理をこのセル内に展開する（本番ファイルには触れない）。

In [ ]:
import numpy as np, xgboost as xgb, json
from sklearn.metrics import roc_auc_score
from src.tools.train_xgb import (_EXCLUDE_COLS, _MARKET_FEAT_COLS,
                                 _popularity_to_base_margin)

t0 = time.time()

# ── 期間分割（検証期の一部を early stopping 用に使う） ───────────
VAL_END = '2026-02-28'          # 学習の early stopping 用
EVAL_START = '2026-03-01'       # ★真のout-of-sample（フィルタ検証はここだけ使う）

train_df = df[df['date'] <= TRAIN_END].copy()
val_df   = df[(df['date'] >= TEST_START) & (df['date'] <= VAL_END)].copy()
eval_df  = df[df['date'] >= EVAL_START].copy()
print(f'学習 {len(train_df):,}行 / 早期停止用 {len(val_df):,}行 / 検証 {len(eval_df):,}行')

# ── 特徴量列（本番と同じ規則: 除外列 + f_popularity を除く数値列）──
feat_cols = [c for c in df.columns
             if c not in _EXCLUDE_COLS and c not in _MARKET_FEAT_COLS
             and df[c].dtype in ('float64', 'int64', 'float32', 'int32')]
print(f'特徴量数: {len(feat_cols)}')

def _prep(d):
    d = d.copy()
    d['_n_horses'] = d.groupby('race_id')['horse_num'].transform('count')
    pop = d['f_popularity'].fillna(d['_n_horses'] / 2)
    bm = _popularity_to_base_margin(pop, d['_n_horses'])
    X = d[feat_cols].fillna(5.0)
    return d, X, bm

train_df, X_tr, bm_tr = _prep(train_df)
val_df,   X_va, bm_va = _prep(val_df)
eval_df,  X_ev, bm_ev = _prep(eval_df)
y_tr, y_va = train_df['is_fukusho'], val_df['is_fukusho']

pos_rate = y_tr.mean()
spw = round((1 - pos_rate) / max(pos_rate, 0.01), 2)

dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=feat_cols); dtr.set_base_margin(bm_tr)
dva = xgb.DMatrix(X_va, label=y_va, feature_names=feat_cols); dva.set_base_margin(bm_va)

params = {'objective': 'binary:logistic', 'max_depth': 6, 'learning_rate': 0.05,
          'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10,
          'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': spw,
          'eval_metric': 'logloss', 'seed': 42, 'nthread': -1}
booster = xgb.train(params, dtr, num_boost_round=500,
                    evals=[(dva, 'val')], early_stopping_rounds=50, verbose_eval=100)

# ⚠ output_margin=True 必須（無いと2重sigmoidになる。2026-07-27に本番で発覚したバグ）
va_margin = booster.predict(dva, output_margin=True)
print(f'\n✅ 学習完了  Val AUC: {roc_auc_score(y_va, 1/(1+np.exp(-va_margin))):.4f}')
print(f'⏱ 所要 {(time.time()-t0)/60:.1f} 分')

booster.save_model(os.path.join(OUT_DIR, 'backtest_model.pkl'))
with open(os.path.join(OUT_DIR, 'backtest_feature_cols.json'), 'w') as f:
    json.dump({'feature_cols': feat_cols, 'residual': True, 'train_end': TRAIN_END}, f)
print(f'   検証用モデルを保存: {OUT_DIR}/backtest_model.pkl（本番ファイルは無変更）')

## セル4: 検証期の推論 → rl_rank 算出

検証期（2026-03以降）の全馬を推論し、レース内でのAI評価順位（rl_rank）を出す。

さらに `history.db` から **確定人気** と **単勝配当** を結合する
（`win_odds` は0%充足のため使えないが、単勝ROIの計算には
「勝ったか」＋「勝った時の配当」があれば足りる）。

In [ ]:
t0 = time.time()

dev = xgb.DMatrix(X_ev, feature_names=feat_cols); dev.set_base_margin(bm_ev)
eval_df['raw_margin'] = booster.predict(dev, output_margin=True)

# レース内でmargin降順 → rl_rank（1位が最高評価）
eval_df['rl_rank'] = eval_df.groupby('race_id')['raw_margin'] \
                            .rank(ascending=False, method='first').astype(int)

# ── history.db から確定人気・着順・単勝配当を結合 ──────────────
conn = sqlite3.connect(HIST)
hh = pd.read_sql_query(
    'SELECT race_id, horse_num, popularity, place, tansho_payout FROM horse_history', conn)
conn.close()
hh['horse_num'] = pd.to_numeric(hh['horse_num'], errors='coerce')
eval_df['horse_num'] = pd.to_numeric(eval_df['horse_num'], errors='coerce')

m = eval_df.merge(hh, on=['race_id', 'horse_num'], how='left', suffixes=('', '_h'))
print(f'結合前 {len(eval_df):,}行 → 結合後 {len(m):,}行')

# 人気が有効な馬のみ対象（99や欠損は除外）
m = m[(m['popularity'].notna()) & (m['popularity'] >= 1) & (m['popularity'] < 99)].copy()
m['popularity'] = m['popularity'].astype(int)
m['won'] = (m['place'] == 1)
m['payout'] = m['tansho_payout'].fillna(0)
# 勝ったのに配当が取れていない行は集計から外す（回収率を過小評価しないため）
bad = m['won'] & (m['payout'] <= 0)
print(f'⚠ 勝ったが配当欠損のため除外: {bad.sum()}頭')
m = m[~bad].copy()

print(f'\n✅ 検証対象: {len(m):,}頭 / {m["race_id"].nunique():,}レース')
print(f'   期間: {m["date"].min()} 〜 {m["date"].max()}')
print(f'   全体の単勝回収率（全馬に賭けた場合）: {m["payout"].sum()/(len(m)*100)*100:.1f}%')
print(f'⏱ 所要 {(time.time()-t0):.0f} 秒')

## セル5: ★ マトリクス作成と検証（メインの結果）

検証期をさらに前半・後半に分け、
**前半でマトリクスを作り、後半で成績を測る**（out-of-sample）。

本番と同じ閾値（N≥50・回収率≥120%でboost、<50%でsuppress）を使う。

In [ ]:
MIN_N, BOOST_ROI, SUPPRESS_ROI = 50, 120.0, 50.0

def band(r):
    r = int(r)
    return '1' if r == 1 else '2-3' if r <= 3 else '4-5' if r <= 5 else '6-9' if r <= 9 else '10+'

m['pop_band'] = m['popularity'].map(band)
m['rl_band']  = m['rl_rank'].map(band)

# 検証期をさらに半分に割る（前半=マトリクス作成 / 後半=成績測定）
dates = sorted(m['date'].unique())
split = dates[len(dates) // 2]
build_df = m[m['date'] < split]
test_df  = m[m['date'] >= split]
print(f'マトリクス作成: {build_df["date"].min()}〜{build_df["date"].max()} '
      f'({build_df["race_id"].nunique():,}R)')
print(f'成績測定      : {test_df["date"].min()}〜{test_df["date"].max()} '
      f'({test_df["race_id"].nunique():,}R)\n')

def make_matrix(d):
    g = d.groupby(['pop_band', 'rl_band']).agg(
        n=('won', 'size'), wins=('won', 'sum'), pay=('payout', 'sum')).reset_index()
    g['win_rate'] = g['wins'] / g['n'] * 100
    g['roi'] = g['pay'] / (g['n'] * 100) * 100
    return g

mx = make_matrix(build_df)
def judge(row):
    if row['n'] < MIN_N: return None
    if row['roi'] >= BOOST_ROI: return 'boost'
    if row['roi'] < SUPPRESS_ROI: return 'suppress'
    return None
mx['flag'] = mx.apply(judge, axis=1)

print('=== マトリクス（前半で作成）===')
bands = ['1', '2-3', '4-5', '6-9', '10+']
print(f'{"人気＼RL":<10}' + ''.join(f'{("RL"+b):>16}' for b in bands))
look = {(r['pop_band'], r['rl_band']): r for _, r in mx.iterrows()}
for pb in bands:
    line = f'{pb+"人気":<10}'
    for rb in bands:
        c = look.get((pb, rb))
        line += f'{"-":>16}' if c is None else f'{f"{c.roi:.0f}%/{c.n}":>16}'
    print(line)

print('\n=== boost / suppress 判定されたセル ===')
for _, r in mx[mx['flag'].notna()].sort_values('roi', ascending=False).iterrows():
    print(f'  市場{r["pop_band"]:>4}人気 × RL{r["rl_band"]:<4} : '
          f'N={int(r["n"]):>4} 勝率{r["win_rate"]:>5.1f}% 回収率{r["roi"]:>6.1f}%  → {r["flag"].upper()}')

# ── 後半で成績測定（★本命の結果）──────────────────────────────
flag_map = {(r['pop_band'], r['rl_band']): r['flag'] for _, r in mx.iterrows() if r['flag']}
test_df = test_df.copy()
test_df['flag'] = [flag_map.get((p, r), 'neutral')
                   for p, r in zip(test_df['pop_band'], test_df['rl_band'])]

print('\n' + '=' * 62)
print('★★★ out-of-sample 検証結果（後半期間で測定）★★★')
print('=' * 62)
print(f'{"判定":<10}{"点数":>8}{"的中":>7}{"勝率":>8}{"回収率":>9}{"損益":>12}')
for k in ['boost', 'neutral', 'suppress']:
    g = test_df[test_df['flag'] == k]
    if len(g) == 0: continue
    inv, pay = len(g) * 100, g['payout'].sum()
    print(f'{k:<10}{len(g):>8,}{int(g["won"].sum()):>7}{g["won"].mean()*100:>7.1f}%'
          f'{pay/inv*100:>8.1f}%{f"¥{pay-inv:,.0f}":>12}')

bt = test_df[test_df['flag'] == 'boost']
if len(bt) >= 30:
    print(f'\n【boost馬だけ買った場合】')
    print(f'  点数 {len(bt):,} / 的中 {int(bt["won"].sum())} / '
          f'回収率 {bt["payout"].sum()/(len(bt)*100)*100:.1f}%')
    # 日別のばらつき（1日単位で見て負ける日がどれくらいあるか）
    daily = bt.groupby('date').apply(
        lambda d: d['payout'].sum() / (len(d) * 100) * 100)
    print(f'  開催日数 {len(daily)}日 / プラスの日 {int((daily>=100).sum())}日 '
          f'({(daily>=100).mean()*100:.0f}%)')
    print(f'  日別回収率の中央値 {daily.median():.1f}% / 最悪の日 {daily.min():.1f}%')
else:
    print(f'\n⚠ boost該当が{len(bt)}点しかなく判断できません')

if len(bt) == 0:
    print('\n⚠ boost該当が0点でした。考えられる原因:')
    print('   ① モデルにエッジが無い（AI順位が市場と独立した情報を持っていない）')
    print('   ② 前半期間のセルがどれもN<50 または 回収率<120%')
    print('   → 上の「マトリクス（前半で作成）」を見て、回収率が高いセルが')
    print('      そもそも存在するかを確認してください')


## セル6: より良いフィルタの探索

データ量が増えたので、**現在の閾値より良い設定があるか**を探す。

**⚠ ここで見つかった設定をそのまま採用しないこと。**
同じデータで最適化すると過学習する（実際に7/27の作業でオッズ帯の絞り込みが
学習期で良く見えたのに検証期で悪化した）。**必ず後半期間で再確認すること。**

In [ ]:
print('=== ① 閾値を変えた場合（前半で判定→後半で測定）===')
print(f'{"MIN_N":>7}{"BOOST_ROI":>11}{"→点数":>9}{"回収率":>9}')
best = []
for min_n in [30, 50, 80, 120]:
    for b_roi in [110, 120, 130, 150]:
        f = {(r['pop_band'], r['rl_band']) for _, r in mx.iterrows()
             if r['n'] >= min_n and r['roi'] >= b_roi}
        g = test_df[[(p, r) in f for p, r in zip(test_df['pop_band'], test_df['rl_band'])]]
        if len(g) < 30: continue
        roi = g['payout'].sum() / (len(g) * 100) * 100
        best.append((roi, min_n, b_roi, len(g)))
        print(f'{min_n:>7}{b_roi:>11}{len(g):>9,}{roi:>8.1f}%')
if best:
    best.sort(reverse=True)
    r, mn, br, n = best[0]
    print(f'\n  最良: MIN_N={mn}, BOOST_ROI={br} → {n:,}点 回収率{r:.1f}%')
    print('  ⚠ ただしこれは後半データを見て選んだ値。採用するなら別期間で再確認すること')

print('\n=== ② 帯の切り方を変えた場合（RL帯を細かく）===')
def band2(r):
    r = int(r)
    return str(r) if r <= 6 else '7-9' if r <= 9 else '10+'
for d in (build_df, test_df):
    d['pop_b2'] = d['popularity'].map(band2)
    d['rl_b2']  = d['rl_rank'].map(band2)
g2 = build_df.groupby(['pop_b2', 'rl_b2']).agg(
    n=('won', 'size'), pay=('payout', 'sum')).reset_index()
g2['roi'] = g2['pay'] / (g2['n'] * 100) * 100
f2 = {(r['pop_b2'], r['rl_b2']) for _, r in g2.iterrows()
      if r['n'] >= MIN_N and r['roi'] >= BOOST_ROI}
t2 = test_df[[(p, r) in f2 for p, r in zip(test_df['pop_b2'], test_df['rl_b2'])]]
if len(t2) >= 30:
    print(f'  細分化した帯でのboost: {len(t2):,}点 '
          f'回収率{t2["payout"].sum()/(len(t2)*100)*100:.1f}%')
    print(f'  （現行の帯: {len(bt):,}点 回収率'
          f'{bt["payout"].sum()/(len(bt)*100)*100:.1f}%）')
else:
    print(f'  該当{len(t2)}点で判断不能')

print('\n=== ③ 結果サマリ（このセッションの結論に使う）===')
if len(bt) >= 30:
    roi_b = bt['payout'].sum() / (len(bt) * 100) * 100
    print(f'  検証規模     : {test_df["race_id"].nunique():,}レース / {len(test_df):,}頭')
    print(f'  boost該当    : {len(bt):,}点（現在の本番検証は76点）')
    print(f'  boost回収率  : {roi_b:.1f}%')
    print(f'  判定         : ' +
          ('✅ 100%超。フィルタは有効と考えられる' if roi_b >= 100
           else '❌ 100%未満。現在の本番設定は再考が必要'))

out_csv = os.path.join(OUT_DIR, 'backtest_result.csv')
test_df[['date','race_id','horse_num','horse_name','popularity','rl_rank',
         'pop_band','rl_band','flag','won','payout']].to_csv(out_csv, index=False)
print(f'\n  明細を保存: {out_csv}')

## 実行後の報告方法

次のセッションで結果を共有する際は、**セル5とセル6の出力をそのまま貼り付ける**。

特に重要なのは以下:
- boost判定の **点数・回収率**（現在の本番検証は76点しかない）
- **プラスの日の割合**（現在は4日中2日＝50%）
- boost判定された**セルの顔ぶれ**が現在の本番（市場6-9人気×RL4-5 / 市場2-3人気×RL4-5）と一致するか

### 判定の目安

| 結果 | 意味 |
|---|---|
| boost回収率 **120%以上** かつ N>500 | フィルタは本物の可能性が高い。エア馬券から移行を検討できる |
| boost回収率 **100〜120%** | 有望だが控除率を考えると薄い。継続観察 |
| boost回収率 **100%未満** | 現在の本番設定を見直す必要がある（今の76点は偶然だった可能性） |

### 注意事項

- 本番ファイル（`xgb_fukusho_model.pkl` 等）は**一切変更していない**。週末の運用に影響しない
- 検証用モデルは `data/backtest/` にあり、本番では読まれない
- `horse_features.csv` はセル2で再生成される（本番の再学習でも使うファイルなので、
  次回の本番再学習時はそのまま使える）
- 過去には `win_odds` が0%充足のため、**オッズ帯フィルタ（2〜30倍）は適用していない**。
  本番より緩い条件での検証になっている点は差し引いて解釈すること